# 07 · Studi Kasus Pasang Surut Indonesia — Bab 8

*Pengantar Deep Learning untuk Meteorologi* · Kanada Kurniawan

Notebook pendamping Bab 8: pipeline end-to-end prediksi pasang surut dengan contoh station Cilacap (GLOSS #291). Mencakup tiga opsi data (sample sintetik / data nyata hasil unduh / sintetis cepat), windowing, baseline persistence/klimatologi, model MLP/LSTM/GRU, walk-forward, evaluasi per horizon, dan simulasi pengisian gap data. Sel terakhir juga meregenerasi Gambar 8.2 & 8.3 seperti di buku.

## 1. Setup & Pilihan Data

Tiga cara memuat data. Default: **Opsi A** (sample CSV sintetik yang sudah di-commit di repo, ~1 tahun hourly). Untuk data nyata: jalankan `python scripts/download_ioc.py --source ioc --code cili --days 30 --output data/raw/cili_30d.csv` lalu gunakan **Opsi B**.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import tensorflow as tf

np.random.seed(42)
tf.random.set_seed(42)

# --- Opsi A: sample CSV yang sudah ada di repo (default, out-of-the-box) ---
seri = pd.read_csv(
    "data/sample/cili_1y_hourly.csv",
    parse_dates=["time"],
).set_index("time")["tinggi"].astype(float)

# --- Opsi B: data nyata hasil unduh scripts/download_ioc.py (uncomment) ---
# seri = pd.read_csv(
#     "data/raw/cili_30d.csv",
#     parse_dates=["time"],
# ).set_index("time")["tinggi"].astype(float)

# --- Opsi C: sintetis cepat (untuk coba cepat tanpa unduh) ---
# t = pd.date_range("2024-01-01", periods=365*24, freq="h", tz="UTC")
# jam = np.arange(len(t), dtype=float)
# tinggi = (0.40*np.sin(2*np.pi*jam/12.42 + 0.7)
#           + 0.10*np.sin(2*np.pi*jam/12.00 + 1.1)
#           + 0.20*np.sin(2*np.pi*jam/23.93 + 2.3)
#           + 0.10*np.sin(2*np.pi*jam/25.82 + 0.5)
#           + 0.05*np.random.randn(len(t))).round(3)
# seri = pd.Series(tinggi, index=t, name="tinggi")

# Seri bersih untuk analisis (isi gap pendek dengan interpolasi)
ser = seri.interpolate(limit=6).dropna().values
times = seri.interpolate(limit=6).dropna().index

print(seri.head())
print("nilai hilang:", int(seri.isna().sum()), "dari", len(seri))

## 2. Spektrum (Gambar 8.1)

FFT untuk melihat periode dominan — pembacaan cepat tipe pasang.

In [ ]:
fft = np.fft.rfft(ser - ser.mean())
freq = np.fft.rfftfreq(len(ser), d=1.0)
period = np.where(freq > 0, 1/freq, np.inf)
power = np.abs(fft)**2
m = (period > 6) & (period < 100)
plt.figure(figsize=(7, 3.5))
plt.plot(period[m], power[m], color="#4a90e2")
plt.axvline(12.42, color="#e0893d", ls="--", label="12,42 jam (M2)")
plt.axvline(24.84, color="#27ae60", ls="--", label="24,84 jam (K1)")
plt.xscale("log"); plt.xlabel("Periode (jam)"); plt.ylabel("Daya")
plt.title("Spektrum frekuensi"); plt.legend(); plt.tight_layout(); plt.show()

## 3. Windowing & Split

`w` dalam jam; horizon `h` dalam jam (24 = 1 hari). Gunakan split kronologis (Bab 5 §5.5).

In [ ]:
def buat_window(deret, w=168, h=24):
    X, y = [], []
    for i in range(len(deret) - w - h + 1):
        X.append(deret[i:i+w])
        y.append(deret[i+w:i+w+h])
    return np.array(X), np.array(y)

X, y = buat_window(ser, w=168, h=24)
y = y[:, 0]  # target = nilai satu langkah horizon ke depan
print("X", X.shape, "y", y.shape)

n = len(X)
ntr, nva = int(n*0.7), int(n*0.15)
Xtr, ytr = X[:ntr], y[:ntr]
Xva, yva = X[ntr:ntr+nva], y[ntr:ntr+nva]
Xte, yte = X[ntr+nva:], y[ntr+nva:]
print("train", Xtr.shape, "val", Xva.shape, "test", Xte.shape)

## 4. Baseline (persistence & klimatologi)

In [ ]:
def mae(a, b):
    return float(np.mean(np.abs(a - b)))

def rmse(a, b):
    return float(np.sqrt(np.mean((a - b)**2)))

# persistence untuk h=24: tinggi air 24 jam SEBELUM target.
# y[0] sesuai ser[168+24-1]; persistence = ser[168-1+24] dari posisi y[i].
offs = 168 + 24 - 1
persist_test = np.array([ser[ntr + nva + k + offs - 24] for k in range(len(yte))])
base = np.full(len(yte), float(np.mean(ytr)))
print(f"MAE persistence h=24: {mae(yte, persist_test):.4f}")
print(f"MAE klimatologi    : {mae(yte, base):.4f}")
print(f"RMSE persistence h=24: {rmse(yte, persist_test):.4f}")

## 5. Model: MLP, LSTM, GRU (Kode 8.2)

In [ ]:
def buat_model(kind, w=168, f=1):
    if kind == "mlp":
        m = tf.keras.Sequential([
            tf.keras.layers.Dense(32, activation="relu", input_shape=(w*f,)),
            tf.keras.layers.Dense(16, activation="relu"),
            tf.keras.layers.Dense(1)])
    elif kind == "lstm":
        m = tf.keras.Sequential([
            tf.keras.layers.LSTM(16, input_shape=(w, f)),
            tf.keras.layers.Dense(1)])
    else:
        m = tf.keras.Sequential([
            tf.keras.layers.GRU(16, input_shape=(w, f)),
            tf.keras.layers.Dense(1)])
    m.compile(optimizer="adam", loss="mse", metrics=["mae"])
    return m

hasil = {}
pred_lstm = None
for kind in ["mlp", "lstm", "gru"]:
    Xtr_m = Xtr.reshape(len(Xtr), -1) if kind == "mlp" else Xtr
    Xva_m = Xva.reshape(len(Xva), -1) if kind == "mlp" else Xva
    Xte_m = Xte.reshape(len(Xte), -1) if kind == "mlp" else Xte
    m = buat_model(kind, w=168)
    m.fit(Xtr_m, ytr, validation_data=(Xva_m, yva), epochs=20, batch_size=32, verbose=0)
    p = m.predict(Xte_m, verbose=0).ravel()
    hasil[kind] = {"mae": mae(yte, p), "rmse": rmse(yte, p)}
    if kind == "lstm":
        pred_lstm = p
    print(f"{kind.upper():>4}: MAE={hasil[kind]['mae']:.4f}  RMSE={hasil[kind]['rmse']:.4f}")

mae_persist = mae(yte, persist_test)
print(f"\nBaseline persistence: MAE={mae_persist:.4f}")
print("Skill score (relatif thd persistence):")
for k, v in hasil.items():
    ss = 1 - v["mae"] / mae_persist
    print(f"  {k.upper():>4}: SS = {ss:+.3f}")

## 6. Plot 7 Hari dari Test Set (latihan membaca plot)

In [ ]:
n_awal = Xte.shape[0] - 7*24
plt.figure(figsize=(10, 3.4))
plt.plot(yte[n_awal:], label="aktual", lw=1.3)
plt.plot(pred_lstm[n_awal:], label="prediksi LSTM", lw=1.1, ls="--", alpha=0.85)
plt.xlabel("Indeks test (bukan waktu asli)")
plt.ylabel("Tinggi muka air (m, relatif)")
plt.title("Prediksi vs aktual — 7 hari terakhir pada test set")
plt.legend(); plt.tight_layout(); plt.show()
print("Catatan: sumbu-x di sini adalah indeks pada test set (bukan waktu).")
print("Untuk plot dengan waktu asli (seperti di buku), lihat sel Bagian 8.")

## 7. Simulasi Pengisian Gap (Kode 8.3)

Sembunyikan 24 jam dari data, latih tanpa gap, prediksi, lalu ukur MAE terhadap nilai asli.

In [ ]:
from math import isnan

pos = 1500  # contoh posisi gap (diubah saat eksperimen)
garis = ser.copy()
garis[pos:pos+24] = np.nan
print(f"Gap 24 jam pada pos {pos}; baris tersembunyi: {sum(isnan(x) for x in garis[pos:pos+24])}")
print("Langkah: latih model pada subset utuh, prediksi window akhir sebelum gap,")
print("         bandingkan dgn nilai asli yang disembunyikan.")
print("Lihat Kode 8.3 di master.md untuk alur lengkap + MAE imputasi.")

## 8. Regenerasi Gambar 8.2 & 8.3 (seperti di buku)

Sel di bawah ini melatih satu model (MLP atau fallback persistence bila TF tidak
tersedia), lalu memplot:

- **Gambar 8.2** — prediksi vs aktual untuk **7 hari terakhir data** (sumbu-x = waktu asli).
- **Gambar 8.3** — residu per amplitudo aktual (kiri) dan per fase siklus M2 (kanan).

Plot di buku dihasilkan oleh skrip `scripts/generate_figures.py`; sel ini adalah
versi notebook-nya.

In [ ]:
from pathlib import Path

# Latih ulang pada seluruh deret (bukan test split) agar prediksi bisa dipasang
# di posisi mana saja — untuk plot 7 hari terakhir, kita pakai model yang
dilatih di paruh pertama deret, lalu prediksi paruh kedua window-by-window.
W = 168
split = int(len(ser) * 0.7)

def make_xy(arr, w):
    X, y = [], []
    for i in range(len(arr) - w):
        X.append(arr[i:i+w])
        y.append(arr[i+w])
    return np.array(X), np.array(y)

Xall, yall = make_xy(ser[:split], W)
model_full = tf.keras.Sequential([
    tf.keras.layers.Dense(32, activation="relu", input_shape=(W,)),
    tf.keras.layers.Dense(16, activation="relu"),
    tf.keras.layers.Dense(1),
])
model_full.compile(optimizer="adam", loss="mse", metrics=["mae"])
model_full.fit(Xall, yall, epochs=15, batch_size=32, verbose=0)

# Prediksi rekursif di paruh kedua: setiap langkah pakai window terakhir prediksi
history = list(ser[:split])
pred_half = []
for t in range(split, len(ser)):
    win = np.array(history[-W:], dtype=float).reshape(1, -1)
    yhat = float(model_full.predict(win, verbose=0).ravel()[0])
    pred_half.append(yhat)
    history.append(ser[t])  # pakai aktual (1-step ahead; untuk multi-step ganti dengan yhat)
pred_half = np.array(pred_half)
print(f"Prediksi 1-step-ahead pada paruh kedua: {len(pred_half)} baris")
print(f"MAE vs aktual: {mae(ser[split:], pred_half):.4f} m")

In [ ]:
# --- Gambar 8.2: prediksi vs aktual, 7 hari terakhir data ---
HARI = 7
N = HARI * 24
t_idx = times[-N:]
aktual = ser[-N:]
# prediksi selaras: pred_half[-N:] cocok dengan ser[-N:]
pred_win = pred_half[-N:]

fig, ax = plt.subplots(figsize=(10, 3.6))
ax.plot(t_idx, aktual, label="aktual", lw=1.4, color="#1f4e79")
ax.plot(t_idx, pred_win, label="prediksi MLP", lw=1.1, ls="--", alpha=0.85, color="#e0893d")
ax.set_xlabel("Waktu (UTC)")
ax.set_ylabel("Tinggi muka air (m, relatif)")
ax.set_title(f"Prediksi vs aktual — {HARI} hari terakhir (Cilacap, data sample)")
ax.legend(loc="upper right")
ax.grid(alpha=0.25)
fig.autofmt_xdate()
fig.tight_layout()
plt.show()

# Simpan ke figures/ agar build PDF bisa pakai (opsional).
out_dir = Path("figures")
out_dir.mkdir(parents=True, exist_ok=True)
fig.savefig(out_dir / "fig-8-2-forecast-7hari.png", dpi=160)
print(f"Simpan {out_dir / 'fig-8-2-forecast-7hari.png'}")

In [ ]:
# --- Gambar 8.3: residu per amplitudo (kiri) dan fase M2 (kanan) ---
residu = pred_win - aktual
fase = (np.arange(N) % int(12.42)) / 12.42

fig, axes = plt.subplots(1, 2, figsize=(11, 3.6))
ax = axes[0]
ax.scatter(aktual, residu, s=10, alpha=0.55, color="#4a90e2")
ax.axhline(0, color="#333", lw=0.8, ls="--")
ax.set_xlabel("Tinggi aktual (m)")
ax.set_ylabel("Residu (prediksi − aktual), m")
ax.set_title("Residu vs amplitudo")
ax.grid(alpha=0.25)

ax = axes[1]
ax.scatter(fase, residu, s=10, alpha=0.55, color="#27ae60")
ax.axhline(0, color="#333", lw=0.8, ls="--")
ax.set_xlabel("Fase dalam siklus M2 (0 = awal siklus)")
ax.set_ylabel("Residu (prediksi − aktual), m")
ax.set_title("Residu vs fase pasang")
ax.grid(alpha=0.25)

fig.suptitle(f"Residu {HARI} hari terakhir — Cilacap, data sample, MLP",
             fontsize=10, y=1.02)
fig.tight_layout()
plt.show()

fig.savefig(out_dir / "fig-8-3-residu.png", dpi=160, bbox_inches="tight")
print(f"Simpan {out_dir / 'fig-8-3-residu.png'}")

## 9. Latihan Mini

1. Ganti data sample dengan data nyata dari `scripts/download_ioc.py` (Cilacap atau station lain) dan jalankan ulang.
2. Uji `w ∈ {24, 72, 168}` pada h=24; buat tabel MAE.
3. Bangun model *direct* untuk h ∈ {24, 72, 168}; plot MAE per horizon.
4. Hitung skill score `SS = 1 − MAE_model/MAE_persistence` per horizon (lihat output sel di atas).
5. Simulasikan gap 72 jam dan bandingkan MAE imputasi vs gap 24 jam.
6. (Tantangan) Ulangi pipeline untuk station Ambon (`--code ambon`) atau Bitung (`--code bitu`); bandingkan tipe pasang dan skill score dengan Cilacap.
7. (Tantangan) Di Sel Bagian 8, ganti MLP dengan LSTM/GRU (input shape `(W, 1)` bukan `(W,)`) dan amati perubahan bentuk plot residu Gambar 8.3.